# Guardian of Truth v5 — новые диалоги τ²-bench «как у организаторов» (независимый dev-набор)

**Зачем.** Текстовые ранжировщики сейчас выбираются на 165 ходах из 39 диалогов — этого не хватает, чтобы
отличить ×1.1 от ×1.3, а старые промпты писались по этим же диалогам. Здесь генерируются ~150 новых диалогов
тем же способом, что у организаторов:
- инструкция агента τ² + «Always communicate with the user in Russian…» (дословно из valid);
- политики доменов дословно из valid; база знаний банка переведена на русский, как у организаторов;
- клиент — Qwen3.6-35B-A3B-FP8, агент — слабая модель **Qwen3.5-9B** без рассуждений;
- задачи — только те, которых нет в valid (airline 30, retail 35, telecom 40, banking_knowledge 45).

**Среда:** Colab, **A100 80GB**, High-RAM. Секрет Colab `HF_TOKEN` желателен (скачивание моделей быстрее).

**Оценка:** ≈ 1.5–2 ч ≈ **11–15 CU** (A100 ≈ 7.5 CU/ч): установка и модели ~20 мин, перевод базы ~3 мин,
диалоги ~60–80 мин. RAM ≈ 20 ГБ, диск ≈ 70 ГБ.

**Чекпоинты:** всё на Диске в `MyDrive/guardian_of_truth/v5_tau2gen/`. Готовые модели, перевод и домены
пропускаются. Если сессия упала — запусти всё сверху ещё раз, повторится только незаконченный домен.

**Результат:** `MyDrive/guardian_of_truth/v5_tau2gen/results_v5_tau2gen.zip` — пришли его Claude.

In [1]:
# 1. Диск, папки, GPU  (⏱ <1 мин)
import os, sys, json, time, subprocess, shutil, glob
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/guardian_of_truth/v5_tau2gen'); RUNS = OUT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'],
                     capture_output=True, text=True).stdout)
print(sys.version)

Mounted at /content/drive
name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
# 2. vLLM 0.19.1 (как на платформе) + τ²-bench в отдельном окружении, чтобы зависимости не мешали vLLM  (⏱ 4–7 мин)
import importlib.metadata as md
try:
    vllm_version = md.version('vllm')
except md.PackageNotFoundError:
    vllm_version = None
if vllm_version != '0.19.1':
    !pip -q install vllm==0.19.1
print('vllm', md.version('vllm'))

TAU2 = Path('/content/tau2-bench'); TAU2_PY = '/content/tau2env/bin/python'
if not (TAU2 / '.ready').exists():
    if not TAU2.exists():
        !git clone -q https://github.com/sierra-research/tau2-bench.git {TAU2}
    !cd {TAU2} && git checkout -q 4ce7c0397c1eb65c9bbe59aeacfe1ca44a1cd699
    !pip -q install uv
    !uv venv -q --python 3.12 /content/tau2env
    !uv pip install -q --python {TAU2_PY} -e "{TAU2}[knowledge]"
    (TAU2 / '.ready').write_text('ok')
print(subprocess.run([TAU2_PY, '-c', 'import tau2, litellm; print("tau2 ok, litellm", litellm.__version__)'],
                     capture_output=True, text=True))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11

In [3]:
# 3. Модели: клиент Qwen3.6-35B-A3B-FP8, агент Qwen3.5-9B  (⏱ 8–15 мин в первый раз)
from huggingface_hub import snapshot_download
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as error:
    print('HF_TOKEN не найден, качаю без него:', error)
MODELS = {'user': 'Qwen/Qwen3.6-35B-A3B-FP8', 'agent': 'Qwen/Qwen3.5-9B'}
for role, repo in MODELS.items():
    target = Path('/content/models') / role
    if not (target / '.complete').exists():
        t = time.time()
        snapshot_download(repo, local_dir=str(target), allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja',
                                                                       '*.model', '*.tiktoken', 'merges.txt'])
        (target / '.complete').write_text(repo)
        print(f'✓ {role}: {repo} за {time.time() - t:.0f} с')
    print(role, sorted(p.name for p in target.iterdir())[:8], '…')

HF_TOKEN не найден, качаю без него: Secret HF_TOKEN does not exist.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

✓ user: Qwen/Qwen3.6-35B-A3B-FP8 за 102 с
user ['.cache', '.complete', 'chat_template.jinja', 'config.json', 'configuration.json', 'generation_config.json', 'layers-0.safetensors', 'layers-1.safetensors'] …


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✓ agent: Qwen/Qwen3.5-9B за 51 с
agent ['.cache', '.complete', 'chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors-00001-of-00004.safetensors', 'model.safetensors-00002-of-00004.safetensors', 'model.safetensors-00003-of-00004.safetensors'] …


In [4]:
# 4. Код генерации (упакован в ноутбук)  (⏱ <1 мин)
import base64, io, tarfile
CODE_B64 = '...'
GEN = Path('/content/tau2gen'); GEN.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz') as archive:
    archive.extractall(GEN)
print(sorted(p.name for p in GEN.iterdir()))

['organizer_policies.json', 'organizer_tools.json', 'patches.py', 'run_simulations.py', 'serialize.py', 'translate_kb.py', 'valid_tasks.json']


/tmp/ipykernel_1178/1728462950.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(GEN)


In [5]:
# 5. Два сервера vLLM: клиент (порт 8002) и агент (порт 8001), с разбором вызовов инструментов  (⏱ 5–8 мин)
import urllib.request

def start_server(role, port, memory_share, parser='qwen3_coder'):
    log = open(f'/content/vllm_{role}.log', 'w')
    command = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', f'/content/models/{role}',
               '--served-model-name', role, '--port', str(port), '--max-model-len', '131072',
               '--gpu-memory-utilization', str(memory_share), '--max-num-seqs', '64', '--enable-prefix-caching',
               '--language-model-only', '--enable-auto-tool-choice', '--tool-call-parser', parser]
    return subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)

def wait_ready(role, port, process, limit=1200):
    started = time.time()
    while time.time() - started < limit:
        if process.poll() is not None:
            print(open(f'/content/vllm_{role}.log').read()[-4000:]); raise RuntimeError(f'vLLM {role} упал')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models', timeout=5); print(f'✓ {role} готов за {time.time()-started:.0f} с'); return
        except Exception:
            time.sleep(5)
    raise RuntimeError(f'vLLM {role} не поднялся')

SERVERS = {}
SERVERS['user'] = start_server('user', 8002, 0.55); wait_ready('user', 8002, SERVERS['user'])
SERVERS['agent'] = start_server('agent', 8001, 0.33); wait_ready('agent', 8001, SERVERS['agent'])

def chat(port, model, messages, tools=None):
    body = {'model': model, 'messages': messages, 'temperature': 0, 'max_tokens': 200,
            'chat_template_kwargs': {'enable_thinking': False}}
    if tools: body['tools'] = tools
    request = urllib.request.Request(f'http://127.0.0.1:{port}/v1/chat/completions', data=json.dumps(body).encode(),
                                     headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(request, timeout=300).read())['choices'][0]['message']

probe_tool = [{'type': 'function', 'function': {'name': 'get_user_details', 'description': 'Get user details.',
               'parameters': {'type': 'object', 'properties': {'user_id': {'type': 'string'}}, 'required': ['user_id']}}}]
for role, port in (('agent', 8001), ('user', 8002)):
    answer = chat(port, role, [{'role': 'user', 'content': 'Найди пользователя sara_doe_496.'}], probe_tool)
    print(role, '→', answer.get('tool_calls') or answer.get('content'))

✓ user готов за 325 с
✓ agent готов за 115 с
agent → [{'id': 'chatcmpl-tool-94f3ebfe91fe6e68', 'type': 'function', 'function': {'name': 'get_user_details', 'arguments': '{"user_id": "sara_doe_496"}'}}]
user → [{'id': 'chatcmpl-tool-97edd83199603b10', 'type': 'function', 'function': {'name': 'get_user_details', 'arguments': '{"user_id": "sara_doe_496"}'}}]


In [6]:
# 6. База знаний банка → русский (как у организаторов), кэш на Диске  (⏱ 2–4 мин в первый раз)
DOCS = TAU2 / 'data/tau2/domains/banking_knowledge/documents'
!{sys.executable} {GEN}/translate_kb.py --docs {DOCS} --url http://127.0.0.1:8002/v1 --model user --cache {OUT}/kb_ru.json
sample = json.loads(sorted(DOCS.glob('*.json'))[5].read_text()); print(sample['title']); print(sample['content'][:300])

документов 698, уже переведено 0, переводим 698
  50/698 за 36 с
  100/698 за 49 с
  150/698 за 62 с
  200/698 за 72 с
  250/698 за 85 с
  300/698 за 96 с
  350/698 за 107 с
  400/698 за 118 с
  450/698 за 133 с
  500/698 за 148 с
  550/698 за 159 с
  600/698 за 168 с
  650/698 за 177 с
  698/698 за 185 с
готово: переведено 698/698
Внутренний документ: Закрытие счетов личных сбережений
## Требования перед закрытием

Убедитесь в соблюдении всех следующих условий перед закрытием:
- Если применяется комиссия за досрочное закрытие, остаток по счету должен быть не меньше суммы комиссии; в противном случае остаток по счету (current_holdings) должен составлять $0. Комиссия списывается не


In [8]:
# ХОТФИКС к уже запущенной сессии: недостающие зависимости τ² + новый patches.py (набор банка как у организаторов)
# Вставь в новую ячейку после ячейки 6, запусти, затем перезапусти ячейку 7 и дальше 8–10.
import base64, gzip, subprocess
from pathlib import Path
!apt-get -qq install -y portaudio19-dev > /dev/null
!uv pip install -q --python /content/tau2env/bin/python websockets scipy "elevenlabs>=1.54.0" pyaudio
Path('/content/tau2gen/patches.py').write_bytes(gzip.decompress(base64.b64decode('H4sIAK0txmoC/7VZ3W4bxxW+51NMqIssHWoDGMgNHaaQY9kQYksGSfnGNhYrckUttNwldpdS1MCAfpImhZMaLQq0F20D9KK56A0tWzFtyQ7gF+jyFfIk/c6ZWe7s8qdy0xKCuDtz5vzPmW8Oy+Vy8n3yJnmeXCRnyevxY5Gcin99+fZ0ecvx2ztVMf56fIz5pzTzGg+n48fJmQD9KBkm5xh4lowETeNlJMYn46Px8fgQ44rbK5C9Ivohlr+Wi9+A4BkPjJIXyZAE8JJTYSSnGHtOkkTyE/7t2Z7bqZilUvIDkYnxEdFhasgsz8bHGJrHcQijRrWSED8f/l2QFiNowPqNT5JX49/gHboPsZKUPU6GpOcbMB9BzJCVePvPFW/fPohEO+j1Br7btmNH7Lvxjoh3HDGInFC4vmgMosi1fVN85jh9EQeBJ3y750RVSP7H2/NrExXISVKYYDHfsAVk6TIpBeWOQDSqYgj6j2Ap5kll9stZXtmfD/8I/ZI/Jc/Z1lNJB9Ne4vvsPZH8ANILAScgSuMTFkMvCOm3v9K0omHohUUjCpd0AqeDEkIDUOycXc+hgZdlaBCxp+yqV5wb0B5W/G5OQJAPHL+nEHbI6YOMIaWHHP6Xgu09BA056SXnYU10Q/gUM9CIid+IaMfxvMpE/1Mx0eFMkJ26kz67bkWOHbZ3rK3e1Y+q2nvH8SMHprCAa9JqaK82wjElL7F9QQ8vyBdszEvyF5QEEZTFIpXn5Cs9hMlLqCeEMf6OVWbCc5WxcWj7kYdMsna3zP4BiyYZQ0T9hJcPkfLJ37CS4vI1uaM2pTlZCqlIf96hP5Lx5NWncPWxksRqXb9z9SMZR4reGZIOhoy/EuPvYOtTNoK282t4jfYg0xxfoyBfjJ8IGaOvKJUoN/QtdKHvm/Pxk1LO85SEgtSi3fsXqjFYcj7+lhlT0oHHMWUChtkr9uCqNAVElE+sLxxC9WQk7H7fO7D6dtzecSKjUhOcj1ReiJyy0mBjXjCjisDjM+Jxwb6heXIkJ9sxK8EKckaNZPWRM2apXC6XtsOgJyxrexAPQseyhNvrB2EsbN8PYjt2Az8qldRYvBM6dsf1u6VSY7PZXFtZt1Zura63rMbm7VVRF0b5l5UQfm7bnifssDvoOX6MwbUb+NcOOkRQ5kyb/pRtvyMCCAlFx45t2rADJxLO53Y79g6EHZECB+RYpBXpQPpEB1Hs9DhdOoGAtVm2iiBUL27shDQwVzI49cxyZeKQzeZqg/3RhEPIv6WlJXHb9rsDu+uUloXy0H4IzjCYNcGMH2uuqfIo6IQN18TunpPOiAgW7MLM/WDgdUzwYycq/7GrnOWe7Xp46O8EviP8QW/LCTMX9sOgM2jHcon4j37rQrhPqh0Eg1A8KLl+FIdYT3lxSdexh6Bps++0XduD0bvY1cJ4f2lpqdnauIuv96uC3lqNlfXmzdVGNrKx2VreuLnc/HTj7iqNVkQUwy1TgQ0dk3ydBuFWY3W1tbZ+i0LwX54aYLbaaqyt3lu5bd1baYBri7jtOQicHVlIUsrWCPG1OHyoWPGgH9VEx4V36+KLR4VMWcJujXYtHCTLn4gvyjL7LISj14/LNWGaJvJb8qyJ+3h9+IgdLGGJUHABpVWr+iXLC9q7kDbZmeZtDBiVUqnUcbaFrplRIcGkXY1zmfclr69NcjsEZegzkZEzK2Uoq5Ka0nJBMl9HwklmacFAfTKZ2PS8nlxGQZu8SOJtMaOcUFYh7yakppxbW2+2GpufttY21jO9FxDBOYtmPxDlB/4Dv4yHaR3yRlP9sjy1kxcZTIT8z4rc3gCbAlsCRudHeFUQul3Xx56oF2bNrhNbXS/Ysj0rnbG6A7fjeK7vRKnbQGXHcWikfJBBoSwU5aq4aXuRUykGt8TvZJai1NgaEGVxCtbEFtXiumJCtiLWU4mSys0W1idPFc2lWVGU4qdFm2oIIlvhwGGqd3IJFk5zzQcQ6MeJsUcWxS4I6dBFASOJ+gtFUH9X8dOob6zeXNm83bJurjWaLZVFd1abTTyZ7cCPKffroliiUhVDp+uiIoTWlu3vQkkMxKHroCIX1UWhS/6q0M/hbJjC4Gbu3aNWgIoIVBFxfcBQ0ZTRSv7AAIIB728ZrmrodcTYldA5YeKzmfqMH789l7VMQkQlNBXFIBcXgSGDOGkXoaojBYgZpdXE+Eua5MsCsBBrtjVwvY5MN3mPAFSltQzwz4mhus0Nx98Ihj2HfJvjWwm5SMJBaayg66HUEHDwhAD8qcTqdJkZEtzNK8h4kWgIO/L9hS8VOT1HZhoz/mawxYnWCXBM+5GZhnvXD/Y9p9N10nR0/D03DHxCQVUxSYbLcjEnK6ye+zlIUrbGLdh6h4bohtDkoF9HPFY8r0V+lFNz0M47fFLeNyjIOeaVS9sgA6sU/ywdZk4yM9ueHUWFGTKGhZKhxnwTF2hYFZmTLumJvA5a1eXNjaPSjS3LiBxvuyo6W1Wxu8Vbz+q7fS5VPMIbQhuitJy8ajzpEw36TmhUzAnvzlYlTwBZplWUgwpUHJq9Kq+LXJYfm7EupzDW5N5V6U/z0pzCV837U0MPqaxnK9Kne3aIQh8bExUIzdanVmfBkzDLAuLqE0itZzzvNjbu3G01rRtrDfEhLhKeJ+uJ2evgDJXATVW/I9XA0FoII8Gw8vfyPqpaBi+4HUOV74lqajyR11PuI0x0ksVLaqYpBIDrd+wwnclsQAC0oq2boBxMANuID/pOvUwU5cqstRzDd1tMUdRW0N5gakVSyUEZS6/Jdb2KmdrMPACjrwaYCcKu7bu/xv2FcOglMI223NiTSaL2W7o/cb5GDhVUuwPw2yUgjxkcvXU6YTXW0E5xMCm7xHs4u4sJlt+SBVBk/RJlpoey3Z36dlHdg5h3quFZeK02ZHPDZrJzU+3NfAoWDan8f0Vy5v7PZeZK1EQqjU6JKkTAREpa03EyFoVOpUicHWFakpj5dNeh8JxtBBL9TZUrjXiqAyW7P3SJVD2vV4RhzqifJBtnP2kt8id8P+YW24XCqcC77d0JRpW5zfiU7Mzw6Z8nXdX5+DTXjc53No8UZiMlC7iNwJnUciT7h8KQLTkJAN9IaKpKMzfyqa8GPHZCrCvmOyAxU3ekQiF0CdGGi8GhMy9PYWRATCXX9DlVKWm5EUGM0zHItbLwbFNHhW5juA3reUCCVAAq2S2iHeDo4NtuZPQDz20fWMGeE4a4EqnmxIzrxPf5oE+dc8VWs+wBa6fi/N85KA7c7+XeyogBNARUJ32NCV+9xWGoYCJ3aWI051qRhVO/w4UD38cNnHcGXdv4oXjf5kG1l3KtiFl3avYqcuMyt+qU1tJ4G7JhQuGs5oH9FTvs4qS7cmV3n540xrJZUs/u2JfmkWFPzmyZk2xTbllZTjNHGMbHn376qdX4K+YRtw137Ih5sloZN0lbLqBV2X7JkUCtIt/7kuBhVmftiLpb0i5KeKNMQ+WcnkzkRtwwIiNmapdvtxWxNPW4qGenjCjXlPFTC2vKktzonCMoa+bxXg76jm+7VoQK2rNzm1qyZOqHj3K8ZvXo0k+uP3c/gq3kCNPtVAg083Dx2JF5rorNjDQ1J6P62TO1UxiTz1ieNh1zP19ctgZN02U/zhHxQgy+6GdW4vAj/SIjPpYyPqF68vGHk5fT9KfXtJbMb3Lq04V2oD6VNZrk6KLeTkqxqG5XSv8GI3e7OL0eAAA=')))
check = subprocess.run(['/content/tau2env/bin/python', '-c',
                        'import sys, json; sys.path.insert(0, "/content/tau2gen"); import tau2.registry, tau2.run; '
                        'from patches import apply_patches, check_banking_tools; '
                        'apply_patches(json.load(open("/content/tau2gen/organizer_policies.json"))); '
                        'print("инструменты банка:", check_banking_tools())'],
                       cwd='/content/tau2-bench', capture_output=True, text=True)
print(check.stdout[-1500:], check.stderr[-3000:])

инструменты банка: ['KB_search_bm25', 'KB_search_dense', 'call_discoverable_agent_tool', 'change_user_email', 'get_credit_card_accounts_by_user', 'get_credit_card_transactions_by_user', 'get_current_time', 'get_referrals_by_user', 'get_user_information_by_email', 'get_user_information_by_id', 'get_user_information_by_name', 'give_discoverable_user_tool', 'grep', 'list_discoverable_agent_tools', 'log_verification', 'transfer_to_human_agents', 'unlock_discoverable_agent_tool']
 2026-10-07 11:34:04.640 | WARNING  | tau2.utils.utils:<module>:15 - No .env file found
2026-10-07 11:34:04.641 | INFO     | tau2.utils.utils:<module>:28 - Using data directory from source: /content/tau2-bench/data
2026-10-07 11:34:11.069 | INFO     | tau2.utils.llm_utils:<module>:102 - LiteLLM: Cache is disabled
2026-10-07 11:34:11.253 | DEBUG    | tau2.registry:<module>:281 - Registering default components...
2026-10-07 11:34:11.275 | INFO     | tau2.voice.utils.utils:<module>:26 - Found 4 continuous noise files 

In [9]:
# 7. Пробный диалог (1 задача airline) — проверка всей цепочки до большого прогона  (⏱ 2–4 мин)
SMOKE = Path('/content/smoke'); SMOKE.mkdir(exist_ok=True)
common = ['--exclude', f'{GEN}/valid_tasks.json', '--policies', f'{GEN}/organizer_policies.json',
          '--agent-url', 'http://127.0.0.1:8001/v1', '--agent-model', 'agent',
          '--user-url', 'http://127.0.0.1:8002/v1', '--user-model', 'user']
result = subprocess.run([TAU2_PY, f'{GEN}/run_simulations.py', '--domain', 'airline', '--tasks', '1', '--seed', '99',
                         '--concurrency', '1', '--out', str(SMOKE), *common], cwd=str(TAU2), capture_output=True, text=True)
print(result.stdout[-3000:]); print(result.stderr[-3000:])
smoke = json.loads((SMOKE / 'airline.json').read_text())
for message in smoke['simulations'][0]['messages'][:8]:
    print(message['role'], '|', (message.get('content') or '')[:150].replace(chr(10), ' '), '|',
          [c['name'] for c in message.get('tool_calls') or []])

                             │
│ nl: None                                                                     │
│ communicate: {'note': 'No communicate_info to evaluate'}                     │
│ action: None                                                                 │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Successfully completed all simulations!
To review the simulations, run: tau2 view
╭───────────────────────── Agent Performance Metrics ──────────────────────────╮
│   ═══ Overview ═══                                                           │
│   Total Simulations         1                                                │
│   Total Tasks               1                                                │
│                                                                              │
│   ═══ Reward Metrics ═══                                                  

In [10]:
# 8. Большой прогон: домены по очереди, готовые пропускаются  (⏱ 60–80 мин)
TASKS = {"airline": 30, "retail": 35, "telecom": 40, "banking_knowledge": 45}
for domain, count in TASKS.items():
    if (RUNS / f'{domain}.json').exists():
        print('✓ уже есть', domain); continue
    t = time.time()
    process = subprocess.run([TAU2_PY, f'{GEN}/run_simulations.py', '--domain', domain, '--tasks', str(count),
                              '--concurrency', '32', '--out', str(RUNS), *common], cwd=str(TAU2),
                             capture_output=True, text=True)
    print(process.stdout[-1500:]); print(process.stderr[-1500:])
    print(f'{domain}: {time.time() - t:.0f} с, файл есть: {(RUNS / (domain + ".json")).exists()}', flush=True)

                       │
│   ═══ DB Match ═══                                                           │
│   🗄️  DB Match              ✓ 18 / ✗ 8 (69.2%)                               │
│                                                                              │
│   ═══ Authentication ═══                                                     │
│      Not Checked            30                                               │
│                                                                              │
│   ═══ Termination ═══                                                        │
│   🛑 Normal Stop            26 (👤 26 / 🤖 0)                                │
│   ⏱️  Max Steps             4                                                │
│                                                                              │
│   ═══ LLM Judge Review ═══                                                   │
│   🤖 Agent Errors           0 errors                                         │
│      

In [11]:
# 9. Ходы в формате соревнования + сверка формата с valid  (⏱ 1–2 мин)
sys.path.insert(0, str(GEN))
from serialize import examples_frame, render_tools
examples = examples_frame(RUNS)
examples.to_parquet(OUT / 'examples_v5.parquet', index=False)
print(examples.groupby('domain').agg(dialogs=('dialog', 'nunique'), turns=('id', 'size'),
                                     text_turns=('n_calls', lambda s: int((s == 0).sum()))))
organizer_tools = json.loads((GEN / 'organizer_tools.json').read_text())
policies = json.loads((GEN / 'organizer_policies.json').read_text())
for domain in TASKS:
    run = json.loads((RUNS / f'{domain}.json').read_text())
    setup = next(iter(run['agent_setups'].values()))
    ours = render_tools(setup['tools'])
    same_names = sorted(l.split(' — ')[0] for l in ours.splitlines() if l.startswith('- ')) == \
                 sorted(l.split(' — ')[0] for l in organizer_tools[domain].splitlines() if l.startswith('- '))
    print(f'{domain}: политика как у организаторов: {policies[domain] in setup["system_prompt"]}, '
          f'инструменты те же: {same_names}, схема дословно: {ours == organizer_tools[domain]}')

                   dialogs  turns  text_turns
domain                                       
airline                 30    585         388
banking_knowledge       44    714         183
retail                  26    503         318
telecom                 40   1008         788
airline: политика как у организаторов: True, инструменты те же: True, схема дословно: False
retail: политика как у организаторов: True, инструменты те же: True, схема дословно: True
telecom: политика как у организаторов: True, инструменты те же: True, схема дословно: True
banking_knowledge: политика как у организаторов: True, инструменты те же: True, схема дословно: True


In [12]:
# 10. Архив результатов → пришли его Claude  (⏱ <1 мин)
archive = OUT / 'results_v5_tau2gen.zip'
!cd {OUT} && rm -f {archive} && zip -q -r {archive} runs examples_v5.parquet kb_ru.json
print(archive, f'{archive.stat().st_size / 1e6:.1f} МБ')
for server in SERVERS.values():
    server.terminate()

/content/drive/MyDrive/guardian_of_truth/v5_tau2gen/results_v5_tau2gen.zip 41.8 МБ
